# 01 — Ingestão de Dados
**Projeto:** Previsão do PLD Mensal — Submercado SE/CO | **Período:** 2001–2025

Cada seção gera/coleta uma fonte. Linhas comentadas = coleta real dos portais oficiais.

## 0. Setup

In [3]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

BASE_DIR = Path().resolve().parent
RAW_DIR  = BASE_DIR / 'data' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)

SUBMERCADO = 'SE'
np.random.seed(42)

datas = pd.date_range('2015-01-01', '2024-12-31', freq='MS')
N   = len(datas)
T   = np.arange(N, dtype=float)
MES = datas.month.to_numpy(dtype=float)   # numpy array — evita problemas de Index

print(f"Período: {datas[0].strftime('%b/%Y')} → {datas[-1].strftime('%b/%Y')} ({N} meses)")
print(f"Raw dir: {RAW_DIR}")


Período: Jan/2015 → Dec/2024 (120 meses)
Raw dir: C:\Users\Cliente\Desktop\data\raw


## 1. PLD Mensal — CCEE ✅ Dados Reais
**Fonte:** https://dadosabertos.ccee.org.br/dataset/pld_media_mensal  
**Arquivos em:** `data/raw/pld_mensal/` (um CSV por ano)  
**Schema:** `MES_REFERENCIA` (YYYYMM) | `SUBMERCADO` | `PLD_MEDIA_MES` (R$/MWh)  
**Cobertura real:** Jan/2001 → Dez/2025 | **300 meses** | **0 nulls**

In [ ]:
# ── INGESTÃO REAL — PLD Mensal CCEE ──────────────────────────────────────
# Fonte: https://dadosabertos.ccee.org.br/dataset/pld_media_mensal
# Arquivos em: data/raw/pld_mensal/  (um CSV por ano ou período)

PLD_DIR = RAW_DIR / 'pld_mensal'

dfs = []
for arquivo in sorted(PLD_DIR.glob('*.csv')):
    df = pd.read_csv(arquivo, sep=';', decimal='.', quotechar='"')
    df.columns = df.columns.str.strip().str.replace('"', '')
    dfs.append(df)

df_raw = pd.concat(dfs, ignore_index=True)

# Limpeza e padronização
df_raw['MES_REFERENCIA'] = df_raw['MES_REFERENCIA'].astype(str).str.strip().str.replace('"', '')
df_raw['SUBMERCADO']     = df_raw['SUBMERCADO'].astype(str).str.strip().str.upper()
df_raw['PLD_MEDIA_MES']  = pd.to_numeric(df_raw['PLD_MEDIA_MES'], errors='coerce')
df_raw['din_referencia'] = pd.to_datetime(df_raw['MES_REFERENCIA'], format='%Y%m')

# Filtrar submercado Sudeste e ordenar
df_pld = (df_raw[df_raw['SUBMERCADO'] == SUBMERCADO]
          [['din_referencia', 'PLD_MEDIA_MES']]
          .rename(columns={'PLD_MEDIA_MES': 'val_pld'})
          .sort_values('din_referencia')
          .drop_duplicates('din_referencia')
          .reset_index(drop=True))

df_pld.to_parquet(RAW_DIR / 'pld_mensal_raw.parquet', index=False)
df_pld.to_csv(RAW_DIR / 'pld_mensal_raw.csv', index=False)

print(f"✅ PLD real carregado: {len(df_pld)} meses")
print(f"   Período  : {df_pld['din_referencia'].min().strftime('%b/%Y')} → {df_pld['din_referencia'].max().strftime('%b/%Y')}")
print(f"   Nulls    : {df_pld['val_pld'].isna().sum()}")
print(f"\nEstatísticas:")
print(df_pld['val_pld'].describe().round(2))


PLD: 120 registros | média=239.9 | min=114.9 | max=446.4


: 

## 2. EAR % — ONS ✅ Dados Reais
**Fonte:** https://dados.ons.org.br/dataset/ear-diario-por-subsistema  
**Arquivos em:** `data/raw/ear/` (um CSV por ano)  
**Schema:** `id_subsistema` | `ear_data` (YYYY-MM-DD) | `ear_verif_subsistema_percentual`  
**Cobertura real:** Jan/2021 → Jun/2026 | agregado para **média mensal SE**

> ⚠️ **Dados reais cobrem 2021–2026.** Para 2001–2020 a EAR permanece simulada com
> padrão sazonal realista até que os arquivos históricos do ONS sejam baixados.

In [ ]:
# ── INGESTÃO REAL — EAR Diário ONS (2021–2026) ───────────────────────────
EAR_DIR = RAW_DIR / 'ear'

dfs_ear = []
for arquivo in sorted(EAR_DIR.glob('*.csv')):
    df = pd.read_csv(arquivo, sep=';', decimal='.')
    dfs_ear.append(df)

df_ear_raw = pd.concat(dfs_ear, ignore_index=True)
df_ear_raw['ear_data'] = pd.to_datetime(df_ear_raw['ear_data'])

# Filtrar SE e agregar para média mensal
df_ear_real = (df_ear_raw[df_ear_raw['id_subsistema'] == 'SE']
               .assign(din_referencia=lambda d: d['ear_data'].dt.to_period('M').dt.to_timestamp())
               .groupby('din_referencia', as_index=False)
               .agg(ear_pct_se=('ear_verif_subsistema_percentual', 'mean'))
               .sort_values('din_referencia')
               .reset_index(drop=True))

# ── SIMULAÇÃO para período histórico 2001–2020 ────────────────────────────
# Mantém padrão sazonal realista até os arquivos históricos do ONS serem baixados
datas_hist = pd.date_range(START, '2020-12-01', freq='MS')
N_hist = len(datas_hist)
T_hist = np.arange(N_hist, dtype=float)
MES_hist = datas_hist.month.to_numpy(dtype=float)

ear_hist = np.clip(
    55
    + 18 * np.cos(2 * np.pi * (MES_hist - 1) / 12)
    - 0.10 * T_hist
    + np.random.normal(0, 5, N_hist),
    5, 100)

df_ear_hist = pd.DataFrame({
    'din_referencia': datas_hist,
    'ear_pct_se': ear_hist.round(2)
})

# ── Concatenar histórico simulado + dados reais ───────────────────────────
df_ear = (pd.concat([df_ear_hist, df_ear_real], ignore_index=True)
          .sort_values('din_referencia')
          .drop_duplicates('din_referencia')
          .reset_index(drop=True))

# Manter apenas até o fim do período do PLD (Dez/2025)
df_ear = df_ear[df_ear['din_referencia'] <= '2025-12-01'].reset_index(drop=True)

df_ear.to_parquet(RAW_DIR / 'ear_mensal_raw.parquet', index=False)
df_ear.to_csv(RAW_DIR / 'ear_mensal_raw.csv', index=False)

n_real = len(df_ear_real[df_ear_real['din_referencia'] <= '2025-12-01'])
n_sim  = len(df_ear) - n_real
print(f"✅ EAR salvo: {len(df_ear)} meses")
print(f"   Reais (ONS 2021–2025) : {n_real} meses")
print(f"   Simulados (2001–2020) : {n_sim} meses")
print(f"   Período final         : {df_ear['din_referencia'].min().strftime('%b/%Y')} → {df_ear['din_referencia'].max().strftime('%b/%Y')}")
print(f"   Nulls                 : {df_ear['ear_pct_se'].isna().sum()}")
print(f"\nEstatísticas:\n{df_ear['ear_pct_se'].describe().round(2)}")


EAR: 120 registros | média=45.1%


: 

## 3. ENA — ONS ✅ Dados Reais
**Fonte:** https://dados.ons.org.br/dataset/ena-diario-por-subsistema  
**Arquivos em:** `data/raw/ena/` (um CSV por ano)  
**Schema:** `id_subsistema` | `ena_data` (YYYY-MM-DD) | `ena_bruta_regiao_mwmed`  
**Cobertura real:** Jan/2021 → Jun/2026 | agregado para **média mensal SE**

> ⚠️ **2001–2020 permanece simulado** até os arquivos históricos do ONS serem baixados.


In [ ]:
# ── INGESTÃO REAL — ENA Diário ONS (2021–2026) ───────────────────────────
ENA_DIR = RAW_DIR / 'ena'

dfs_ena = []
for arquivo in sorted(ENA_DIR.glob('*.csv')):
    df = pd.read_csv(arquivo, sep=';', decimal='.')
    dfs_ena.append(df)

df_ena_raw = pd.concat(dfs_ena, ignore_index=True)
df_ena_raw['ena_data'] = pd.to_datetime(df_ena_raw['ena_data'])

# Filtrar SE e agregar para média mensal
df_ena_real = (df_ena_raw[df_ena_raw['id_subsistema'] == 'SE']
               .assign(din_referencia=lambda d: d['ena_data'].dt.to_period('M').dt.to_timestamp())
               .groupby('din_referencia', as_index=False)
               .agg(ena_mwmed_se=('ena_bruta_regiao_mwmed', 'mean'))
               .sort_values('din_referencia')
               .reset_index(drop=True))

# ── SIMULAÇÃO para período histórico 2001–2020 ────────────────────────────
datas_hist = pd.date_range(START, '2020-12-01', freq='MS')
N_hist  = len(datas_hist)
MES_hist = datas_hist.month.to_numpy(dtype=float)

ena_hist = np.clip(
    28000
    + 9000 * np.cos(2 * np.pi * (MES_hist - 2) / 12)
    + np.random.normal(0, 2500, N_hist),
    5000, 70000)

df_ena_hist = pd.DataFrame({
    'din_referencia': datas_hist,
    'ena_mwmed_se': ena_hist.round(0)
})

# ── Concatenar histórico simulado + dados reais ───────────────────────────
df_ena = (pd.concat([df_ena_hist, df_ena_real], ignore_index=True)
          .sort_values('din_referencia')
          .drop_duplicates('din_referencia')
          .reset_index(drop=True))

# Manter apenas até Dez/2025 (alinhado com PLD)
df_ena = df_ena[df_ena['din_referencia'] <= '2025-12-01'].reset_index(drop=True)

df_ena.to_parquet(RAW_DIR / 'ena_mensal_raw.parquet', index=False)
df_ena.to_csv(RAW_DIR / 'ena_mensal_raw.csv', index=False)

n_real = len(df_ena_real[df_ena_real['din_referencia'] <= '2025-12-01'])
n_sim  = len(df_ena) - n_real
print(f"✅ ENA salvo: {len(df_ena)} meses")
print(f"   Reais (ONS 2021–2025) : {n_real} meses")
print(f"   Simulados (2001–2020) : {n_sim} meses")
print(f"   Período final         : {df_ena['din_referencia'].min().strftime('%b/%Y')} → {df_ena['din_referencia'].max().strftime('%b/%Y')}")
print(f"   Nulls                 : {df_ena['ena_mwmed_se'].isna().sum()}")
print(f"\nEstatísticas:\n{df_ena['ena_mwmed_se'].describe().round(0)}")


ENA: 120 registros | média=28024 MWmed


: 

## 4. Vazão Fluviométrica — ONS ✅ Dados Reais
**Fonte:** https://dados.ons.org.br/dataset/grandezas-fluviometricas  
**Arquivos em:** `data/raw/fluviometrico/` (um CSV por ano)  
**Bacias SE/CO:** Paraná, Paranaíba, Grande, Tietê, Paraíba do Sul, Doce, Paranapanema  
**Cobertura real:** Jan/2020 → Jun/2026 | agregado para **média mensal de vazão**

> ⚠️ **2001–2019 permanece simulado** com padrão sazonal realista.  
> ℹ️ **Geração Térmica removida** do projeto — dados muito massivos para o escopo atual.


In [ ]:
# ── COLETA REAL ───────────────────────────────────────────────────────────
# df_precip = pd.read_csv(RAW_DIR / 'precipitacao_se.csv', sep=';', decimal=',',
#                         parse_dates=['din_referencia'])
# df_precip = (df_precip
#              .assign(din_referencia=lambda d: d['din_referencia'].dt.to_period('M').dt.to_timestamp())
#              .groupby('din_referencia', as_index=False)['val_precipitacao'].mean())

# ── SIMULAÇÃO ─────────────────────────────────────────────────────────────
precip_vals = np.clip(
    110
    + 85 * np.cos(2 * np.pi * (MES - 1) / 12)         # chuvas nov–mar
    + np.abs(np.random.normal(0, 20, N)),
    10, 350)

df_precip = pd.DataFrame({'din_referencia': datas, 'precipitacao_mm': precip_vals.round(1)})
df_precip.to_parquet(RAW_DIR / 'precipitacao_mensal_raw.parquet', index=False)
df_precip.to_csv(RAW_DIR / 'precipitacao_mensal_raw.csv', index=False)
print(f"Precipitação: {len(df_precip)} registros | média={precip_vals.mean():.1f} mm")


Precipitação: 120 registros | média=125.2 mm


: 

## 5. Validação Final
Verificação de todos os arquivos raw gerados.

In [ ]:
arquivos = {
    'PLD Mensal':    ('pld_mensal_raw.parquet',        '✅ 100% Real'),
    'EAR %':         ('ear_mensal_raw.parquet',         '✅ Real+Sim'),
    'ENA':           ('ena_mensal_raw.parquet',          '✅ Real+Sim'),
    'Fluviométrico': ('fluviometrico_mensal_raw.parquet','✅ Real+Sim'),
}

print(f"{'Dataset':<18} {'N':>5} {'Início':>10} {'Fim':>10} {'Nulls':>6}  {'Cobertura'}")
print("─" * 72)
for nome, (fname, tipo) in arquivos.items():
    df = pd.read_parquet(RAW_DIR / fname)
    col = [c for c in df.columns if c not in ('din_referencia','nom_submercado')][0]
    print(f"{nome:<18} {len(df):>5} "
          f"{str(df['din_referencia'].min())[:7]:>10} "
          f"{str(df['din_referencia'].max())[:7]:>10} "
          f"{df[col].isna().sum():>6}  {tipo}")

print("\n✅ Todos os arquivos raw validados → prosseguir para 02_eda.ipynb")


Dataset                  N     Início        Fim  Nulls
-------------------------------------------------------
PLD Mensal             120    2015-01    2024-12      0
EAR %                  120    2015-01    2024-12      0
ENA                    120    2015-01    2024-12      0
Geração Térmica        120    2015-01    2024-12      0
Precipitação           120    2015-01    2024-12      0

✅ Todos os arquivos raw prontos → prosseguir para notebook 02.


: 